<a href="https://colab.research.google.com/github/HussainGit-jpg/mib-lab/blob/main/MID_LAB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import tkinter as tk
from tkinter import ttk, messagebox, scrolledtext
import time
import heapq
import math
import threading
from collections import deque



NODE_POSITIONS = {
    "Internet":     (100, 250),
    "Firewall":     (220, 250),
    "WebServer":    (340, 150),
    "AppServer":    (340, 350),
    "DMZ":          (460, 250),
    "InternalNet":  (580, 150),
    "AdminPC":      (580, 350),
    "FileServer":   (700, 250),
    "BackupServer": (820, 150),
    "Database":     (820, 350),
}


NODE_TYPES = {
    "Internet":     "entry",
    "Firewall":     "firewall",
    "WebServer":    "server",
    "AppServer":    "server",
    "DMZ":          "router",
    "InternalNet":  "router",
    "AdminPC":      "workstation",
    "FileServer":   "server",
    "BackupServer": "server",
    "Database":     "target",
}


EDGES = [
    ("Internet",    "Firewall",     9),
    ("Firewall",    "WebServer",    6),
    ("Firewall",    "AppServer",    7),
    ("WebServer",   "DMZ",          4),
    ("AppServer",   "DMZ",          5),
    ("DMZ",         "InternalNet",  6),
    ("DMZ",         "AdminPC",      8),
    ("InternalNet", "FileServer",   3),
    ("InternalNet", "AdminPC",      4),
    ("AdminPC",     "FileServer",   2),
    ("AdminPC",     "Database",     5),
    ("FileServer",  "BackupServer", 3),
    ("FileServer",  "Database",     4),
    ("BackupServer","Database",     2),
]

START_NODE = "Internet"
GOAL_NODE  = "Database"


def build_graph(edges):
    """Build adjacency list from edge list."""
    graph = {node: [] for node in NODE_POSITIONS}
    for u, v, w in edges:
        graph[u].append((v, w))
        graph[v].append((u, w))
    return graph


# ─────────────────────────────────────────────
#  HEURISTIC FUNCTION FOR A* AND HILL CLIMBING
# ─────────────────────────────────────────────
RISK_LEVELS = {
    "Internet":     10,
    "Firewall":      8,
    "WebServer":     7,
    "AppServer":     7,
    "DMZ":           6,
    "InternalNet":   5,
    "AdminPC":       4,
    "FileServer":    3,
    "BackupServer":  2,
    "Database":      0,
}


def heuristic(node):
    """
    Estimated cost from 'node' to Database (goal).
    Based on network risk/distance level.
    Higher risk level = farther from goal.
    """
    return RISK_LEVELS.get(node, 10)


# ─────────────────────────────────────────────
#  ALGORITHM IMPLEMENTATIONS
# ─────────────────────────────────────────────

def bfs(graph, start, goal):
    """Breadth-First Search — explores level by level."""
    t0 = time.perf_counter()
    queue = deque([[start]])
    visited = set([start])
    nodes_expanded = 0

    while queue:
        path = queue.popleft()
        node = path[-1]
        nodes_expanded += 1

        if node == goal:
            cost = path_cost(path, graph)
            return path, cost, nodes_expanded, (time.perf_counter() - t0) * 1000

        for neighbor, _ in graph[node]:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(path + [neighbor])

    return None, float('inf'), nodes_expanded, (time.perf_counter() - t0) * 1000


def dfs(graph, start, goal):
    """Depth-First Search — explores deep first."""
    t0 = time.perf_counter()
    stack = [[start]]
    visited = set()
    nodes_expanded = 0

    while stack:
        path = stack.pop()
        node = path[-1]

        if node in visited:
            continue
        visited.add(node)
        nodes_expanded += 1

        if node == goal:
            cost = path_cost(path, graph)
            return path, cost, nodes_expanded, (time.perf_counter() - t0) * 1000

        for neighbor, _ in graph[node]:
            if neighbor not in visited:
                stack.append(path + [neighbor])

    return None, float('inf'), nodes_expanded, (time.perf_counter() - t0) * 1000


def ucs(graph, start, goal):
    """Uniform Cost Search — finds cheapest cost path."""
    t0 = time.perf_counter()
    # heap: (cost, path)
    heap = [(0, [start])]
    visited = {}
    nodes_expanded = 0

    while heap:
        cost, path = heapq.heappop(heap)
        node = path[-1]

        if node in visited:
            continue
        visited[node] = cost
        nodes_expanded += 1

        if node == goal:
            return path, cost, nodes_expanded, (time.perf_counter() - t0) * 1000

        for neighbor, w in graph[node]:
            if neighbor not in visited:
                heapq.heappush(heap, (cost + w, path + [neighbor]))

    return None, float('inf'), nodes_expanded, (time.perf_counter() - t0) * 1000


def astar(graph, start, goal):
    """A* Search — uses heuristic for smart pathfinding."""
    t0 = time.perf_counter()
    heap = [(heuristic(start), 0, [start])]
    visited = {}
    nodes_expanded = 0

    while heap:
        f, g, path = heapq.heappop(heap)
        node = path[-1]

        if node in visited:
            continue
        visited[node] = g
        nodes_expanded += 1

        if node == goal:
            return path, g, nodes_expanded, (time.perf_counter() - t0) * 1000

        for neighbor, w in graph[node]:
            if neighbor not in visited:
                new_g = g + w
                new_f = new_g + heuristic(neighbor)
                heapq.heappush(heap, (new_f, new_g, path + [neighbor]))

    return None, float('inf'), nodes_expanded, (time.perf_counter() - t0) * 1000


def hill_climbing(graph, start, goal):
    """
    Hill Climbing — greedy local search.
    Always moves to neighbor with lowest heuristic.
    Can get STUCK at local minima.
    """
    t0 = time.perf_counter()
    current = start
    path = [current]
    visited = set([current])
    nodes_expanded = 0

    while current != goal:
        nodes_expanded += 1
        neighbors = [(heuristic(n), n) for n, _ in graph[current] if n not in visited]

        if not neighbors:
            cost = path_cost(path, graph)
            return path, cost, nodes_expanded, (time.perf_counter() - t0) * 1000

        best_h, best_neighbor = min(neighbors)

        # If best neighbor is worse than current, we're stuck
        if best_h >= heuristic(current) and current != start:
            cost = path_cost(path, graph)
            return path, cost, nodes_expanded, (time.perf_counter() - t0) * 1000

        visited.add(best_neighbor)
        path.append(best_neighbor)
        current = best_neighbor

    cost = path_cost(path, graph)
    return path, cost, nodes_expanded, (time.perf_counter() - t0) * 1000


def path_cost(path, graph):
    """Calculate total cost of a path."""
    total = 0
    for i in range(len(path) - 1):
        u, v = path[i], path[i + 1]
        for neighbor, w in graph[u]:
            if neighbor == v:
                total += w
                break
    return total

DEFENDER_BLOCKED = set()

def minimax(graph, node, depth, is_maximizer, visited, goal, alpha=None, beta=None, use_pruning=False):
    """
    Minimax with optional Alpha-Beta Pruning.
    Attacker = Maximizer (wants to reach goal).
    Defender = Minimizer (wants to block attacker).
    Returns (score, path).
    """
    if node == goal:
        return (100, [node])
    if depth == 0:
        return (-heuristic(node), [node])

    neighbors = [n for n, _ in graph[node] if n not in visited and n not in DEFENDER_BLOCKED]
    if not neighbors:
        return (-50, [node])

    if is_maximizer:
        best_score = -math.inf
        best_path = [node]
        a = alpha if alpha is not None else -math.inf
        for neighbor in neighbors:
            score, sub_path = minimax(graph, neighbor, depth - 1, False,
                                      visited | {node}, goal, a, beta, use_pruning)
            if score > best_score:
                best_score = score
                best_path = [node] + sub_path
            if use_pruning:
                a = max(a, best_score)
                if beta is not None and best_score >= beta:
                    break   # Beta cut-off
        return (best_score, best_path)
    else:
        best_score = math.inf
        best_path = [node]
        b = beta if beta is not None else math.inf
        for neighbor in neighbors:
            score, sub_path = minimax(graph, neighbor, depth - 1, True,
                                      visited | {node}, goal, alpha, b, use_pruning)
            if score < best_score:
                best_score = score
                best_path = [node] + sub_path
            if use_pruning:
                b = min(b, best_score)
                if alpha is not None and best_score <= alpha:
                    break
        return (best_score, best_path)

def run_minimax(graph, start, goal, use_pruning=False):
    """Run Minimax (or Alpha-Beta) and return results."""
    global DEFENDER_BLOCKED
    DEFENDER_BLOCKED = {"BackupServer"}

    t0 = time.perf_counter()
    score, path = minimax(graph, start, depth=6, is_maximizer=True,
                          visited=set(), goal=goal,
                          alpha=-math.inf, beta=math.inf,
                          use_pruning=use_pruning)
    elapsed = (time.perf_counter() - t0) * 1000

    cost = path_cost(path, graph)
    nodes_expanded = len(path)
    return path, cost, nodes_expanded, elapsed



BG_DARK     = "#0d1117"
BG_PANEL    = "#161b22"
BG_CARD     = "#21262d"
ACCENT      = "#00d4aa"
ACCENT2     = "#ff6b6b"
ACCENT3     = "#ffd93d"
TEXT_MAIN   = "#e6edf3"
TEXT_DIM    = "#8b949e"
GREEN       = "#3fb950"
RED         = "#f85149"
ORANGE      = "#d29922"

NODE_COLORS = {
    "entry":      "#ff6b6b",
    "firewall":   "#ffd93d",
    "server":     "#4fc3f7",
    "router":     "#ce93d8",
    "workstation":"#80cbc4",
    "target":     "#66bb6a",
}

ALGO_LIST = [
    "BFS",
    "DFS",
    "UCS",
    "A*",
    "Hill Climbing",
    "Minimax",
    "Alpha-Beta Pruning",
]


class CyberSimApp(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("🔐 AI Cyber Attack Path Simulation")
        self.configure(bg=BG_DARK)
        self.state("zoomed")

        self.graph = build_graph(EDGES)
        self.results = {}
        self.highlighted_path = []
        self.node_ovals = {}
        self.edge_lines = {}
        self.selected_algo = tk.StringVar(value="BFS")

        self._build_ui()
        self._draw_graph()



    def _build_ui(self):
        # ── Title Bar ──
        title_frame = tk.Frame(self, bg=BG_DARK)
        title_frame.pack(fill="x", padx=20, pady=(12, 0))

        tk.Label(title_frame, text="🔐 AI CYBER ATTACK PATH SIMULATION",
                 font=("Consolas", 18, "bold"), bg=BG_DARK, fg=ACCENT).pack(side="left")
        tk.Label(title_frame, text=f"Start: {START_NODE}   →   Goal: {GOAL_NODE}",
                 font=("Consolas", 11), bg=BG_DARK, fg=TEXT_DIM).pack(side="right", pady=5)


        main = tk.Frame(self, bg=BG_DARK)
        main.pack(fill="both", expand=True, padx=20, pady=10)


        left = tk.Frame(main, bg=BG_PANEL, width=260, relief="flat", bd=0)
        left.pack(side="left", fill="y", padx=(0, 10))
        left.pack_propagate(False)
        self._build_left_panel(left)


        center = tk.Frame(main, bg=BG_DARK)
        center.pack(side="left", fill="both", expand=True)
        self._build_canvas(center)


        right = tk.Frame(main, bg=BG_PANEL, width=310, relief="flat", bd=0)
        right.pack(side="right", fill="y", padx=(10, 0))
        right.pack_propagate(False)
        self._build_right_panel(right)


        bottom = tk.Frame(self, bg=BG_PANEL, height=200)
        bottom.pack(fill="x", padx=20, pady=(0, 12))
        bottom.pack_propagate(False)
        self._build_table(bottom)

    def _build_left_panel(self, parent):
        tk.Label(parent, text="CONTROL PANEL", font=("Consolas", 10, "bold"),
                 bg=BG_PANEL, fg=ACCENT).pack(pady=(16, 8))


        tk.Frame(parent, bg=ACCENT, height=1).pack(fill="x", padx=12)


        tk.Label(parent, text="Select Algorithm:", font=("Consolas", 9),
                 bg=BG_PANEL, fg=TEXT_DIM).pack(anchor="w", padx=14, pady=(12, 4))

        for algo in ALGO_LIST:
            rb = tk.Radiobutton(parent, text=algo, variable=self.selected_algo,
                                value=algo, font=("Consolas", 9),
                                bg=BG_PANEL, fg=TEXT_MAIN,
                                selectcolor=BG_CARD,
                                activebackground=BG_PANEL, activeforeground=ACCENT,
                                indicatoron=True, bd=0)
            rb.pack(anchor="w", padx=20, pady=2)

        tk.Frame(parent, bg="#30363d", height=1).pack(fill="x", padx=12, pady=12)


        tk.Button(parent, text="▶  RUN SELECTED",
                  font=("Consolas", 10, "bold"),
                  bg=ACCENT, fg=BG_DARK, relief="flat",
                  cursor="hand2", pady=8,
                  command=self._run_selected).pack(fill="x", padx=14, pady=(0, 8))


        tk.Button(parent, text="⚡  RUN ALL ALGORITHMS",
                  font=("Consolas", 10, "bold"),
                  bg=ACCENT3, fg=BG_DARK, relief="flat",
                  cursor="hand2", pady=8,
                  command=self._run_all).pack(fill="x", padx=14, pady=(0, 8))


        tk.Button(parent, text="↺  RESET",
                  font=("Consolas", 10),
                  bg=BG_CARD, fg=TEXT_DIM, relief="flat",
                  cursor="hand2", pady=6,
                  command=self._reset).pack(fill="x", padx=14)

        tk.Frame(parent, bg="#30363d", height=1).pack(fill="x", padx=12, pady=12)


        tk.Label(parent, text="NODE LEGEND", font=("Consolas", 9, "bold"),
                 bg=BG_PANEL, fg=TEXT_DIM).pack(anchor="w", padx=14)

        legend = [
            ("entry",      "Entry Point"),
            ("firewall",   "Firewall"),
            ("server",     "Server"),
            ("router",     "Router/Net"),
            ("workstation","Workstation"),
            ("target",     "Target (Goal)"),
        ]
        for ntype, label in legend:
            row = tk.Frame(parent, bg=BG_PANEL)
            row.pack(anchor="w", padx=14, pady=2)
            tk.Canvas(row, width=14, height=14, bg=BG_PANEL,
                      highlightthickness=0).pack(side="left")
            c = tk.Canvas(row, width=14, height=14, bg=NODE_COLORS[ntype],
                          highlightthickness=0)
            c.pack(side="left")
            tk.Label(row, text=f"  {label}", font=("Consolas", 8),
                     bg=BG_PANEL, fg=TEXT_DIM).pack(side="left")

    def _build_canvas(self, parent):
        tk.Label(parent, text="NETWORK TOPOLOGY",
                 font=("Consolas", 10, "bold"), bg=BG_DARK, fg=TEXT_DIM).pack()

        self.canvas = tk.Canvas(parent, bg=BG_DARK, highlightthickness=0)
        self.canvas.pack(fill="both", expand=True)

    def _build_right_panel(self, parent):
        tk.Label(parent, text="ALGORITHM RESULTS", font=("Consolas", 10, "bold"),
                 bg=BG_PANEL, fg=ACCENT).pack(pady=(16, 8))
        tk.Frame(parent, bg=ACCENT, height=1).pack(fill="x", padx=12)

        self.result_text = scrolledtext.ScrolledText(
            parent, font=("Consolas", 9), bg=BG_CARD, fg=TEXT_MAIN,
            insertbackground=ACCENT, relief="flat", bd=0,
            state="disabled", wrap="word"
        )
        self.result_text.pack(fill="both", expand=True, padx=8, pady=8)


        self.result_text.tag_config("heading",  foreground=ACCENT,  font=("Consolas", 10, "bold"))
        self.result_text.tag_config("label",    foreground=TEXT_DIM, font=("Consolas", 9))
        self.result_text.tag_config("value",    foreground=TEXT_MAIN,font=("Consolas", 9, "bold"))
        self.result_text.tag_config("path",     foreground=ACCENT3,  font=("Consolas", 9))
        self.result_text.tag_config("success",  foreground=GREEN,    font=("Consolas", 9, "bold"))
        self.result_text.tag_config("fail",     foreground=RED,      font=("Consolas", 9, "bold"))
        self.result_text.tag_config("sep",      foreground="#30363d",font=("Consolas", 9))

    def _build_table(self, parent):
        tk.Label(parent, text="📊  COMPARATIVE ANALYSIS TABLE",
                 font=("Consolas", 10, "bold"), bg=BG_PANEL, fg=ACCENT).pack(pady=(8, 4))

        cols = ("Algorithm", "Path Found?", "Total Cost", "Nodes Expanded", "Time (ms)")
        style = ttk.Style()
        style.theme_use("clam")
        style.configure("Custom.Treeview",
                         background=BG_CARD, foreground=TEXT_MAIN,
                         fieldbackground=BG_CARD, rowheight=24,
                         font=("Consolas", 9))
        style.configure("Custom.Treeview.Heading",
                         background=BG_DARK, foreground=ACCENT,
                         font=("Consolas", 9, "bold"), relief="flat")
        style.map("Custom.Treeview",
                  background=[("selected", ACCENT)],
                  foreground=[("selected", BG_DARK)])

        self.table = ttk.Treeview(parent, columns=cols, show="headings",
                                   style="Custom.Treeview", height=5)
        widths = [160, 100, 100, 140, 110]
        for col, w in zip(cols, widths):
            self.table.heading(col, text=col)
            self.table.column(col, width=w, anchor="center")

        self.table.pack(fill="both", expand=True, padx=8, pady=(0, 8))
        self.table.bind("<<TreeviewSelect>>", self._on_table_select)


    def _draw_graph(self):
        self.canvas.update()
        cw = self.canvas.winfo_width()
        ch = self.canvas.winfo_height()

        # Scale node positions to canvas
        xs = [p[0] for p in NODE_POSITIONS.values()]
        ys = [p[1] for p in NODE_POSITIONS.values()]
        min_x, max_x = min(xs), max(xs)
        min_y, max_y = min(ys), max(ys)

        def scale(x, y):
            px = 60 + (x - min_x) / (max_x - min_x + 1) * (cw - 120)
            py = 40 + (y - min_y) / (max_y - min_y + 1) * (ch - 80)
            return px, py

        self.scaled_pos = {n: scale(*p) for n, p in NODE_POSITIONS.items()}

        self.canvas.delete("all")
        self.edge_lines = {}
        self.node_ovals = {}
        self.node_labels = {}


        for u, v, w in EDGES:
            x1, y1 = self.scaled_pos[u]
            x2, y2 = self.scaled_pos[v]
            mid_x = (x1 + x2) / 2
            mid_y = (y1 + y2) / 2
            line = self.canvas.create_line(x1, y1, x2, y2,
                                            fill="#30363d", width=2, tags="edge")
            self.edge_lines[(u, v)] = line
            self.edge_lines[(v, u)] = line
            self.canvas.create_text(mid_x, mid_y - 8, text=str(w),
                                     font=("Consolas", 7), fill=TEXT_DIM)


        r = 26
        for node, (x, y) in self.scaled_pos.items():
            ntype = NODE_TYPES[node]
            color = NODE_COLORS[ntype]
            oval = self.canvas.create_oval(x - r, y - r, x + r, y + r,
                                            fill=color, outline="#30363d",
                                            width=2, tags="node")
            lbl  = self.canvas.create_text(x, y, text=node.replace("Server", "\nServer")
                                            .replace("Net", "\nNet")
                                            .replace("PC", "\nPC"),
                                            font=("Consolas", 7, "bold"),
                                            fill=BG_DARK, tags="nodelabel",
                                            justify="center")
            self.node_ovals[node]  = oval
            self.node_labels[node] = lbl



    def _run_selected(self):
        algo = self.selected_algo.get()
        threading.Thread(target=self._execute_algo, args=(algo,), daemon=True).start()

    def _run_all(self):
        threading.Thread(target=self._execute_all, daemon=True).start()

    def _execute_all(self):
        for algo in ALGO_LIST:
            self._execute_algo(algo, update_table=False)
            time.sleep(0.05)
        self.after(0, self._refresh_table)

    def _execute_algo(self, algo, update_table=True):
        self._reset_highlight()

        g = self.graph
        if algo == "BFS":
            path, cost, exp, ms = bfs(g, START_NODE, GOAL_NODE)
        elif algo == "DFS":
            path, cost, exp, ms = dfs(g, START_NODE, GOAL_NODE)
        elif algo == "UCS":
            path, cost, exp, ms = ucs(g, START_NODE, GOAL_NODE)
        elif algo == "A*":
            path, cost, exp, ms = astar(g, START_NODE, GOAL_NODE)
        elif algo == "Hill Climbing":
            path, cost, exp, ms = hill_climbing(g, START_NODE, GOAL_NODE)
        elif algo == "Minimax":
            path, cost, exp, ms = run_minimax(g, START_NODE, GOAL_NODE, use_pruning=False)
        elif algo == "Alpha-Beta Pruning":
            path, cost, exp, ms = run_minimax(g, START_NODE, GOAL_NODE, use_pruning=True)
        else:
            return

        reached_goal = path is not None and len(path) > 0 and path[-1] == GOAL_NODE

        self.results[algo] = {
            "path": path or [],
            "cost": cost,
            "expanded": exp,
            "time": ms,
            "reached_goal": reached_goal,
        }

        self.after(0, lambda: self._show_result(algo))
        self.after(0, lambda: self._highlight_path(path or []))
        if update_table:
            self.after(0, self._refresh_table)

    def _show_result(self, algo):
        r = self.results[algo]
        path  = r["path"]
        cost  = r["cost"]
        exp   = r["expanded"]
        ms    = r["time"]
        ok    = r["reached_goal"]

        txt = self.result_text
        txt.config(state="normal")


        txt.insert("end", f"\n{'─'*35}\n", "sep")
        txt.insert("end", f" {algo}\n", "heading")
        txt.insert("end", f"{'─'*35}\n", "sep")


        if ok:
            txt.insert("end", " ✅ Goal Reached!\n", "success")
        else:
            txt.insert("end", " ⛔ Did NOT reach goal (stuck)\n", "fail")

        txt.insert("end", " Path:\n", "label")
        txt.insert("end", f"  {' → '.join(path)}\n", "path")


        txt.insert("end", " Total Cost:  ", "label")
        txt.insert("end", f"{cost}\n", "value")
        txt.insert("end", " Nodes Exp.:  ", "label")
        txt.insert("end", f"{exp}\n", "value")
        txt.insert("end", " Time (ms):   ", "label")
        txt.insert("end", f"{ms:.3f}\n", "value")

        if algo == "Hill Climbing" and not ok:
            txt.insert("end", "\n ⚠ LOCAL MINIMUM! Hill Climbing\n", "fail")
            txt.insert("end", "   got stuck — no better neighbor\n", "label")

        if algo in ("Minimax", "Alpha-Beta Pruning"):
            txt.insert("end", "\n 🛡 Defender blocked: BackupServer\n", "label")

        txt.config(state="disabled")
        txt.see("end")

    def _refresh_table(self):
        for row in self.table.get_children():
            self.table.delete(row)

        for algo in ALGO_LIST:
            if algo not in self.results:
                continue
            r = self.results[algo]
            path_str = "Yes ✅" if r["reached_goal"] else "No ⛔"
            cost_str = str(r["cost"]) if r["cost"] != float('inf') else "—"
            self.table.insert("", "end", iid=algo,
                               values=(algo, path_str, cost_str,
                                       r["expanded"], f"{r['time']:.3f}"))

    def _on_table_select(self, event):
        sel = self.table.selection()
        if sel:
            algo = sel[0]
            if algo in self.results:
                self._highlight_path(self.results[algo]["path"])
                self._show_result(algo)



    def _highlight_path(self, path):
        self._reset_highlight()
        if not path:
            return

        self.highlighted_path = path


        for i in range(len(path) - 1):
            u, v = path[i], path[i + 1]
            key = (u, v)
            if key in self.edge_lines:
                self.canvas.itemconfig(self.edge_lines[key], fill=ACCENT, width=4)

        for node in path:
            if node == START_NODE:
                color = ACCENT2
            elif node == GOAL_NODE:
                color = GREEN
            else:
                color = ACCENT
            self.canvas.itemconfig(self.node_ovals[node], outline=color, width=4)

    def _reset_highlight(self):
        for line in self.edge_lines.values():
            self.canvas.itemconfig(line, fill="#30363d", width=2)
        for node, oval in self.node_ovals.items():
            ntype = NODE_TYPES[node]
            self.canvas.itemconfig(oval, outline="#30363d", width=2)
        self.highlighted_path = []

    def _reset(self):
        self._reset_highlight()
        self.results = {}
        for row in self.table.get_children():
            self.table.delete(row)
        self.result_text.config(state="normal")
        self.result_text.delete("1.0", "end")
        self.result_text.config(state="disabled")

    def on_resize(self, event):
        self._draw_graph()
        if self.highlighted_path:
            self._highlight_path(self.highlighted_path)

# if __name__ == "__main__":
#     app = CyberSimApp()
#     app.bind("<Configure>", lambda e: app.after(100, app._draw_graph)
#              if e.widget == app else None)
#     app.mainloop()